In [56]:
import sys

sys.path.append("./CROMA")

In [57]:
from use_croma import PretrainedCROMA
print(PretrainedCROMA)

<class 'use_croma.PretrainedCROMA'>


In [58]:
import inspect

print(inspect.signature(PretrainedCROMA))

(pretrained_path='CROMA_base.pt', size='base', modality='both', image_resolution=120)


In [59]:
print(inspect.getsource(PretrainedCROMA.__init__))

    def __init__(self, pretrained_path='CROMA_base.pt', size='base', modality='both', image_resolution=120):
        """
        NOTE: image_resolution is not the spatial, spectral, or temporal resolution. It is the height and width of the image, in pixels.
        E.g., CROMA was pretrained on 120x120px images, hence image_resolution is 120 by default
        """
        super().__init__()
        # check types
        assert type(pretrained_path) == str, f'pretrained_path must be a string, not {type(pretrained_path)}'
        assert type(size) == str, f'size must be a string, not {type(size)}'
        assert type(modality) == str, f'modality must be a string, not {type(modality)}'
        assert type(image_resolution) == int, f'image_resolution must be an int, not {type(image_resolution)}'

        # check values
        assert size in ['base', 'large'], f'size must be either base or large, not {size}'
        assert image_resolution % 8 == 0, f'image_resolution must be a multiple of

In [60]:
import use_croma

print(use_croma.__file__)

c:\Users\Stefano\Desktop\THESIS\CROMA\./CROMA\use_croma.py


In [61]:
from huggingface_hub import hf_hub_download

checkpoint_path  = hf_hub_download(
    repo_id="antofuller/CROMA",
    filename="CROMA_base.pt",
    local_dir="./CROMA"
)

print(checkpoint_path )

CROMA\CROMA_base.pt


In [62]:
import torch

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

print(type(checkpoint))
print(checkpoint.keys())

<class 'dict'>
dict_keys(['s1_encoder', 's1_GAP_FFN', 's2_encoder', 's2_GAP_FFN', 'joint_encoder'])


In [63]:
s1_state = checkpoint["s1_encoder"]

print(type(s1_state))
print("Number of entries:", len(s1_state))

<class 'collections.OrderedDict'>
Number of entries: 70


In [64]:
for name, tensor in list(s1_state.items())[:20]:
    print(f"{name:60s} {tuple(tensor.shape)}")

linear_input.weight                                          (768, 128)
linear_input.bias                                            (768,)
transformer.layers.0.0.to_qkv.weight                         (2304, 768)
transformer.layers.0.0.to_out.weight                         (768, 768)
transformer.layers.0.0.to_out.bias                           (768,)
transformer.layers.0.0.input_norm.weight                     (768,)
transformer.layers.0.0.input_norm.bias                       (768,)
transformer.layers.0.1.net.0.weight                          (3072, 768)
transformer.layers.0.1.net.0.bias                            (3072,)
transformer.layers.0.1.net.3.weight                          (768, 3072)
transformer.layers.0.1.net.3.bias                            (768,)
transformer.layers.0.1.input_norm.weight                     (768,)
transformer.layers.0.1.input_norm.bias                       (768,)
transformer.layers.1.0.to_qkv.weight                         (2304, 768)
transformer.layers.

In [65]:
name, tensor = next(iter(s1_state.items()))

print("Parameter name:", name)
print("Shape:", tensor.shape)
print("dtype:", tensor.dtype)
print(tensor.flatten()[:10])

Parameter name: linear_input.weight
Shape: torch.Size([768, 128])
dtype: torch.float32
tensor([ 0.0147, -0.0280,  0.0795, -0.0208, -0.0687,  0.0334, -0.0772, -0.0837,
         0.0590, -0.0140])


In [66]:
for key, value in checkpoint.items():
    if isinstance(value, dict):
        n_params = sum(t.numel() for t in value.values() if torch.is_tensor(t))
        print(f"{key:25s} {n_params:,} parameters")
    else:
        print(key, type(value))

s1_encoder                42,614,016 parameters
s1_GAP_FFN                4,723,968 parameters
s2_encoder                85,618,944 parameters
s2_GAP_FFN                4,723,968 parameters
joint_encoder             56,684,544 parameters


In [67]:
model = PretrainedCROMA(
    pretrained_path=checkpoint_path,
    size="base",
    modality="both",
    image_resolution=120
)

Initializing SAR encoder
Initializing optical encoder
Initializing joint SAR-optical encoder


In [68]:
print(type(model))

<class 'use_croma.PretrainedCROMA'>


In [69]:
print(model)


PretrainedCROMA(
  (s1_encoder): ViT(
    (linear_input): Linear(in_features=128, out_features=768, bias=True)
    (transformer): BaseTransformer(
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): Attention(
            (to_qkv): Linear(in_features=768, out_features=2304, bias=False)
            (to_out): Linear(in_features=768, out_features=768, bias=True)
            (input_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (1): FFN(
            (net): Sequential(
              (0): Linear(in_features=768, out_features=3072, bias=True)
              (1): GELU(approximate='none')
              (2): Dropout(p=0.0, inplace=False)
              (3): Linear(in_features=3072, out_features=768, bias=True)
            )
            (input_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          )
        )
      )
      (norm_out): LayerNorm((768,), eps=1e-05, elementwis

In [70]:
'''import torch
from torchview import draw_graph
import os
import shutil
import subprocess

graphviz_bin = r"C:\Program Files\Graphviz\bin"

os.environ["PATH"] += os.pathsep + graphviz_bin

print(shutil.which("dot"))


subprocess.run(["dot", "-V"])
sar = torch.randn(1, 2, 120, 120)

graph = draw_graph(
    model.s1_encoder,
    input_data=(sar, model.attn_bias),
    expand_nested=True,
    depth=4,
    graph_name="CROMA_SAR_Encoder"
)
'''
#graph.visual_graph

'import torch\nfrom torchview import draw_graph\nimport os\nimport shutil\nimport subprocess\n\ngraphviz_bin = r"C:\\Program Files\\Graphviz\x08in"\n\nos.environ["PATH"] += os.pathsep + graphviz_bin\n\nprint(shutil.which("dot"))\n\n\nsubprocess.run(["dot", "-V"])\nsar = torch.randn(1, 2, 120, 120)\n\ngraph = draw_graph(\n    model.s1_encoder,\n    input_data=(sar, model.attn_bias),\n    expand_nested=True,\n    depth=4,\n    graph_name="CROMA_SAR_Encoder"\n)\n'

In [71]:
print("PARAMETERS")

for name, parameter in list(model.named_parameters())[:10]:
    print(name, parameter.shape)

PARAMETERS
s1_encoder.linear_input.weight torch.Size([768, 128])
s1_encoder.linear_input.bias torch.Size([768])
s1_encoder.transformer.layers.0.0.to_qkv.weight torch.Size([2304, 768])
s1_encoder.transformer.layers.0.0.to_out.weight torch.Size([768, 768])
s1_encoder.transformer.layers.0.0.to_out.bias torch.Size([768])
s1_encoder.transformer.layers.0.0.input_norm.weight torch.Size([768])
s1_encoder.transformer.layers.0.0.input_norm.bias torch.Size([768])
s1_encoder.transformer.layers.0.1.net.0.weight torch.Size([3072, 768])
s1_encoder.transformer.layers.0.1.net.0.bias torch.Size([3072])
s1_encoder.transformer.layers.0.1.net.3.weight torch.Size([768, 3072])


In [72]:
print("BUFFERS")

for name, buffer in model.named_buffers():
    print(name, buffer.shape)

BUFFERS


In [73]:
print("STATE DICT")

for name, tensor in list(model.state_dict().items())[:15]:
    print(name, tensor.shape)

STATE DICT
s1_encoder.linear_input.weight torch.Size([768, 128])
s1_encoder.linear_input.bias torch.Size([768])
s1_encoder.transformer.layers.0.0.to_qkv.weight torch.Size([2304, 768])
s1_encoder.transformer.layers.0.0.to_out.weight torch.Size([768, 768])
s1_encoder.transformer.layers.0.0.to_out.bias torch.Size([768])
s1_encoder.transformer.layers.0.0.input_norm.weight torch.Size([768])
s1_encoder.transformer.layers.0.0.input_norm.bias torch.Size([768])
s1_encoder.transformer.layers.0.1.net.0.weight torch.Size([3072, 768])
s1_encoder.transformer.layers.0.1.net.0.bias torch.Size([3072])
s1_encoder.transformer.layers.0.1.net.3.weight torch.Size([768, 3072])
s1_encoder.transformer.layers.0.1.net.3.bias torch.Size([768])
s1_encoder.transformer.layers.0.1.input_norm.weight torch.Size([768])
s1_encoder.transformer.layers.0.1.input_norm.bias torch.Size([768])
s1_encoder.transformer.layers.1.0.to_qkv.weight torch.Size([2304, 768])
s1_encoder.transformer.layers.1.0.to_out.weight torch.Size([768,

In [74]:
p = model.s1_encoder.linear_input.weight

print(type(p))
print(p.requires_grad)

<class 'torch.nn.parameter.Parameter'>
True


In [75]:
print(type(model.attn_bias))
print(model.attn_bias.shape)

<class 'torch.Tensor'>
torch.Size([1, 16, 225, 225])


In [76]:
print(
    any(name == "attn_bias"
        for name, _ in model.named_buffers())
)

print(
    "attn_bias" in model.state_dict()
)

False
False


In [77]:
for name, tensor in list(model.state_dict().items())[:15]:
    print(name, tensor.shape)

s1_encoder.linear_input.weight torch.Size([768, 128])
s1_encoder.linear_input.bias torch.Size([768])
s1_encoder.transformer.layers.0.0.to_qkv.weight torch.Size([2304, 768])
s1_encoder.transformer.layers.0.0.to_out.weight torch.Size([768, 768])
s1_encoder.transformer.layers.0.0.to_out.bias torch.Size([768])
s1_encoder.transformer.layers.0.0.input_norm.weight torch.Size([768])
s1_encoder.transformer.layers.0.0.input_norm.bias torch.Size([768])
s1_encoder.transformer.layers.0.1.net.0.weight torch.Size([3072, 768])
s1_encoder.transformer.layers.0.1.net.0.bias torch.Size([3072])
s1_encoder.transformer.layers.0.1.net.3.weight torch.Size([768, 3072])
s1_encoder.transformer.layers.0.1.net.3.bias torch.Size([768])
s1_encoder.transformer.layers.0.1.input_norm.weight torch.Size([768])
s1_encoder.transformer.layers.0.1.input_norm.bias torch.Size([768])
s1_encoder.transformer.layers.1.0.to_qkv.weight torch.Size([2304, 768])
s1_encoder.transformer.layers.1.0.to_out.weight torch.Size([768, 768])


In [78]:
parameter_names = {
    name for name, _ in model.named_parameters()
}

state_names = set(model.state_dict().keys())

print("Parameters:", len(parameter_names))
print("State entries:", len(state_names))

print("State but not parameters:")
print(state_names - parameter_names)

Parameters: 328
State entries: 328
State but not parameters:
set()


In [79]:
a = model.s1_encoder.linear_input.weight
b = model.state_dict()["s1_encoder.linear_input.weight"]

print(type(a))
print(type(b))

<class 'torch.nn.parameter.Parameter'>
<class 'torch.Tensor'>
